# Encoding Categorical Features

This notebook is the sixth step in the data preparation pipeline, following feature engineering. Its purpose is to convert all **categorical columns** into **numeric form** so that machine-learning algorithms can process them.

**Why encoding is necessary:** Scikit-learn models require all input features to be numeric. Strings like `'True'`/`'False'` or `'C'`/`'D'` cannot be used directly in mathematical operations such as matrix multiplication or gradient computation. Encoding bridges this gap by mapping each text category to a number.

Two encoding strategies are used in this notebook:
1. **Label Encoding**: for binary features (True/False), where we assign 0 and 1.
2. **Ordinal Encoding** — for the `energyClass` feature, which has a natural ordering from most efficient (A+) to least efficient (H). By assigning numbers that respect this order, we allow models to learn the relationship between energy efficiency and rent price.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder


In [2]:
df = pd.read_csv('spitogatos_feat_eng.csv')
df.head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,yearBuilt,locationFloor,totalParkingSpots,apartmentAge,district,is_renovated,rooms_area,bath_area,floor_elevator,floor_view
0,20304396,True,False,3.850148,False,1.0,0.0,True,C,False,...,1975.0,Κέντρο Θεσσαλονίκης,0.0,51.0,1.0,1,3.850148,3.850148,7,7
1,19956518,True,False,3.218876,False,1.0,0.0,True,B,False,...,1982.0,Κέντρο Θεσσαλονίκης,0.0,44.0,1.0,1,3.218876,3.218876,8,8
2,18383741,False,False,4.094345,False,1.0,0.0,True,D,False,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,6,6
3,20242029,True,False,4.094345,False,1.0,0.0,True,F,False,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,8,8
4,20055845,True,False,4.317488,False,1.0,0.0,True,C,False,...,1981.0,Τούμπα,0.0,45.0,4.0,0,8.634976,4.317488,4,4


## Label Encoding for Binary Features

Most of the amenity columns (`airConditioning`, `alarm`, `balcony`, `elevator`, `fireplace`, `furnished`, `garden`, `hasParking`, `isNewDevelopment`, `petsAllowed`, `pool`, `solarHeater`, `storage`, `view`) contain only two unique values: `'True'` and `'False'`.

We use scikit-learn's `LabelEncoder` to automatically map these to 0 and 1. The encoder assigns alphabetically, so `'False'` becomes 0 and `'True'` becomes 1. This is convenient because it means 1 consistently represents "the amenity is present," which is intuitive for model interpretation.

**Why LabelEncoder and not one-hot encoding?** For binary features, one-hot encoding would create two columns per feature, which is redundant, one column is always the exact inverse of the other. Label encoding is more memory-efficient and produces the same information in a single column.

The `.astype(str)` call is a safety measure: if any value is already numeric (e.g., after a previous imputation step), it ensures all values are strings before encoding, preventing type errors.


In [3]:
le_features = ['airConditioning', 'alarm', 'balcony', 'elevator', 'fireplace', 
        'furnished', 'garden', 'hasParking', 'isNewDevelopment', 
        'petsAllowed', 'pool', 'solarHeater', 'storage', 'view']

for feature in le_features:
    le = LabelEncoder()
    df[feature] = le.fit_transform(df[feature].astype(str))


## Ordinal Encoding for `energyClass`

The `energyClass` column is different from the binary features above: it has 11 unique values (`A+`, `A`, `B`, `B+`, `C`, `D`, `E`, `F`, `G`, `H`, `Under issuance`) that follow a **natural ordering** from most energy-efficient to least efficient.

If we used `LabelEncoder` here, it would assign numbers alphabetically — `A+` would get 0, `A` would get 1, but `B+` would get 3 while `B` gets 2, which is backwards (`B+` is more efficient than `B`). This would introduce a non-meaningful ordering that could confuse the model.

Instead, we define a **manual mapping** that respects the true ordinal relationship: `A+ = 0` (best) through `H = 9` (worst), with `Under issuance = 10`. The `.map()` function then replaces each string label with its corresponding number. This encoding preserves the monotonic relationship between energy class and the target, allowing linear models to leverage it as a single numeric feature rather than requiring one-hot encoding.


In [4]:
class_to_number = {
    'A+': 0,
    'A': 1,
    'B+': 2,
    'B': 3,
    'C': 4,
    'D': 5,
    'E': 6,
    'F': 7,
    'G': 8,
    'H': 9,
    'Under issuance': 10
}

df['energyClass'] = df['energyClass'].map(class_to_number)


In [5]:
df.head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,yearBuilt,locationFloor,totalParkingSpots,apartmentAge,district,is_renovated,rooms_area,bath_area,floor_elevator,floor_view
0,20304396,1,0,3.850148,0,1.0,0.0,1,4,0,...,1975.0,Κέντρο Θεσσαλονίκης,0.0,51.0,1.0,1,3.850148,3.850148,7,7
1,19956518,1,0,3.218876,0,1.0,0.0,1,3,0,...,1982.0,Κέντρο Θεσσαλονίκης,0.0,44.0,1.0,1,3.218876,3.218876,8,8
2,18383741,0,0,4.094345,0,1.0,0.0,1,5,0,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,6,6
3,20242029,1,0,4.094345,0,1.0,0.0,1,7,0,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,8,8
4,20055845,1,0,4.317488,0,1.0,0.0,1,4,0,...,1981.0,Τούμπα,0.0,45.0,4.0,0,8.634976,4.317488,4,4


In [6]:
df.to_csv('spitogatos_encode.csv', index=False)
